# Synthetic dataset training run

# Continual Learning â€” Colab Training Notebook

**Before running:**
1. Set runtime to **GPU** (Runtime â†’ Change runtime type â†’ T4 GPU or better)
2. Edit the **Configuration** cell below
3. Run all cells top-to-bottom

Checkpoints and the dataset cache are stored on **Google Drive** and survive session restarts.
If a checkpoint already exists for your `RUN_ID`, training will automatically resume from it.

## Configuration
Edit the values below before running the notebook.

In [3]:
# ============================================================
#  CONFIGURE EVERYTHING HERE
# ============================================================

# Training method â€” choose one:
#   "full_ft"  â€” fine-tune all parameters (catastrophic forgetting baseline)
#   "lora"     â€” parameter-efficient LoRA adapters
#   "smf"      â€” frozen backbone + sparse trainable memory
#   "casm"     â€” versioned memory bank with contradiction-aware routing
METHOD = "casm"

# HuggingFace model (must have approved access)
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Unique experiment name â€” checkpoints are stored under this ID
RUN_ID = "llama1b_synthetic_casm_similarity_23"

# Periods to train â€” full sequence is all four in order
# Each period is one temporal snapshot of the synthetic dataset
PERIODS = ["2018", "2020", "2022", "2024"]  # full run: ["2018", "2020", "2022", "2024"]

# Google Drive folder â€” checkpoints and dataset cache go here
DRIVE_DIR = "/content/drive/MyDrive/continual_learning/synthetic"

# ---- Dataset fraction ----
# Fraction of the dataset to use for both training and evaluation.
# The fraction is applied proportionally and independently to the changed and
# unchanged probe splits, so training and evaluation always cover the same
# examples.  None means use all data; 0.1 means use 10% of each split.
DATASET_FRACTION = None  # synthetic dataset is small (605 facts) â€” use all data

# ---- Core hyperparameters ----
BATCH_SIZE              = 4
GRAD_ACCUM_STEPS        = 4       # effective batch size = 4
LEARNING_RATE           = 5e-4
EPOCHS_PER_PERIOD       = 12     # passes over all passages
MAX_PASSAGES_PER_PERIOD = None    # use all available passages
PRECISION               = "bfloat16"
SEED                    = 42

# ---- Activation capture ----
# Records per-layer hidden-state activations after each period.
# Output saved to periods/<period>/activations.pt and activation_metadata.json
# alongside eval outputs â€” copy to another machine for plotting, no GPU needed.
CAPTURE_ACTIVATIONS = True

# ---- LoRA settings (only used when METHOD == "lora") ----
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ---- SMF settings (only used when METHOD == "smf") ----
# Llama-3.2-1B has 16 transformer layers (indices 0-15)
SMF_MEMORY_SIZE            = 64
SMF_SPARSITY_RATIO         = 0.1
SMF_UPDATE_LAYERS          = [4, 8, 12]  # mid-to-late layers
SMF_REGULARIZATION_WEIGHT  = 0.01
SMF_LEARNING_RATE          = 1e-3

# ---- CASM settings (only used when METHOD == "casm") ----
CASM_NUM_SLOTS               = 32
CASM_ROUTER_HIDDEN_SIZE      = 256
CASM_TOP_K                   = 3
CASM_ROUTER_TEMPERATURE      = 0.1
CASM_SPARSITY_WEIGHT         = 0.0
CASM_OVERLAP_WEIGHT          = 0.01
CASM_BRANCH_ON_CONTRADICTION = False
CASM_MEMORY_SIZE             = 128
CASM_NUM_INJECTION_LAYERS    = 3
CASM_ROUTER_TYPE             = 'similarity'  # 'mlp' = CASMRouter (learned, default), 'similarity' = SimilarityRouter (cosine, no training)

## Setup

In [4]:
# Mount Google Drive for persistent checkpoint and dataset storage
from google.colab import drive
drive.mount("/content/drive")

import os

CHECKPOINT_DIR   = os.path.join(DRIVE_DIR, "checkpoints")
DATASET_CACHE_DIR = os.path.join(DRIVE_DIR, "dataset_cache")
REPO_DIR         = "/content/Continual-Learning"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_CACHE_DIR, exist_ok=True)

print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print(f"Dataset cache:   {DATASET_CACHE_DIR}")

Mounted at /content/drive
Checkpoint dir:  /content/drive/MyDrive/continual_learning/synthetic/checkpoints
Dataset cache:   /content/drive/MyDrive/continual_learning/synthetic/dataset_cache


In [5]:
# Clone the repo (or pull if already cloned this session)
import subprocess

REPO_URL = "https://github.com/athirai-s/Continual-Learning"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned â€” pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")

print("Repo ready.")

Cloning https://github.com/athirai-s/Continual-Learning ...
Cloning into '/content/Continual-Learning'...
Repo ready.


In [6]:
# Install dependencies not pre-installed on Colab
# transformers 5.x is required â€” Colab ships with 4.x by default
!pip install -q "transformers>=5.3.0" "peft>=0.18.1" "trl>=0.29.0" "datasets>=4.6.1" "sentence-transformers>=3.0.0"

# Add the repo to the Python path so imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Dependencies installed. Repo on path.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.0 MB/s eta 0:00:00
Dependencies installed. Repo on path.


## Dataset

Training passages come from `data/augmented/synthetic/<period>.csv` â€” these are
committed directly in the repo, so no download is needed.

Evaluation probes come from `data/probes.json`, which is also committed in the repo.
`SyntheticDataset` loads this file directly â€” no zip extraction needed.
The synthetic dataset contains 605 facts with ~150 changed probes per period boundary.

In [7]:
import os

# Verify augmented training CSVs (data/augmented/synthetic/<period>.csv)
aug_dir = os.path.join(REPO_DIR, "data", "augmented", "synthetic")
for period in PERIODS:
    csv_path = os.path.join(aug_dir, f"{period}.csv")
    assert os.path.exists(csv_path), f"Synthetic CSV missing: {csv_path}"
    print(f"  {period}.csv: OK ({os.path.getsize(csv_path):,} bytes)")

# Verify probes file
probes_json = os.path.join(REPO_DIR, "data", "probes.json")
assert os.path.exists(probes_json), (
    f"probes.json not found at {probes_json}. "
    "Run data/build_probes.py to generate it."
)
print(f"probes.json: OK ({os.path.getsize(probes_json):,} bytes)")
print("All synthetic dataset files present.")


  2018.csv: OK (73,841 bytes)
  2020.csv: OK (74,448 bytes)
  2022.csv: OK (74,093 bytes)
  2024.csv: OK (74,281 bytes)
probes.json: OK (1,247,972 bytes)
All synthetic dataset files present.


## HuggingFace Authentication

Required for gated models like Llama. Log in with a token that has **read** access to
`meta-llama/Llama-3.2-1B`. You can create a token at https://huggingface.co/settings/tokens

In [8]:
from huggingface_hub import login, logout
login()  # will prompt for your HuggingFace access token

## Training

In [9]:
import os

aug_dir = os.path.join(REPO_DIR, "data", "augmented", "synthetic")
all_ok = True
for period in PERIODS:
    path = os.path.join(aug_dir, f"{period}.csv")
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  {period}.csv: {status} ({size:,} bytes)")
    if not exists:
        all_ok = False

if all_ok:
    print("All synthetic CSVs found â€” ready to train.")
else:
    raise FileNotFoundError(
        "One or more synthetic CSVs are missing. "
        "Check that data/augmented/synthetic/<period>.csv files are committed to the repo."
    )


  2018.csv: OK (73,841 bytes)
  2020.csv: OK (74,448 bytes)
  2022.csv: OK (74,093 bytes)
  2024.csv: OK (74,281 bytes)
All synthetic CSVs found â€” ready to train.


In [10]:
# Build TrainConfig from configuration variables set at the top
from training.train_config import TrainConfig

config_kwargs = dict(
    run_id=RUN_ID,
    model_name=MODEL_NAME,
    method=METHOD,
    dataset_name="synthetic",
    precision=PRECISION,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    epochs_per_period=EPOCHS_PER_PERIOD,
    max_passages_per_period=MAX_PASSAGES_PER_PERIOD,
    dataset_fraction=DATASET_FRACTION,
    log_every_n_steps=10,
    eval_after_each_period=True,
    capture_activations=CAPTURE_ACTIVATIONS,
    seed=SEED,
)

if METHOD == "lora":
    config_kwargs.update(
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_target_modules=LORA_TARGET_MODULES,
    )
elif METHOD == "smf":
    config_kwargs.update(
        smf_memory_size=SMF_MEMORY_SIZE,
        smf_sparsity_ratio=SMF_SPARSITY_RATIO,
        smf_update_layers=SMF_UPDATE_LAYERS,
        smf_regularization_weight=SMF_REGULARIZATION_WEIGHT,
        smf_learning_rate=SMF_LEARNING_RATE,
        smf_freeze_backbone=True,
    )
elif METHOD == "casm":
    config_kwargs.update(
        casm_num_slots=CASM_NUM_SLOTS,
        casm_router_hidden_size=CASM_ROUTER_HIDDEN_SIZE,
        casm_top_k=CASM_TOP_K,
        casm_router_temperature=CASM_ROUTER_TEMPERATURE,
        casm_sparsity_weight=CASM_SPARSITY_WEIGHT,
        casm_overlap_weight=CASM_OVERLAP_WEIGHT,
        casm_branch_on_contradiction=CASM_BRANCH_ON_CONTRADICTION,
        casm_memory_size=CASM_MEMORY_SIZE,
        casm_num_injection_layers=CASM_NUM_INJECTION_LAYERS,
        casm_router_type=CASM_ROUTER_TYPE,
    )

cfg = TrainConfig(**config_kwargs)
cfg.validate()

print(f"Method:               {cfg.method}")
print(f"Model:                {cfg.model_name}")
print(f"Periods:              {PERIODS}")
print(f"Batch size:           {cfg.batch_size}")
print(f"Grad accum steps:     {cfg.grad_accum_steps}  (eff. batch = {cfg.batch_size * cfg.grad_accum_steps})")
print(f"Learning rate:        {cfg.learning_rate}")
print(f"Epochs per period:    {cfg.epochs_per_period}")
print(f"Max passages:         {cfg.max_passages_per_period}")
print(f"Dataset fraction:     {cfg.dataset_fraction if cfg.dataset_fraction is not None else 'all (None)'}")
print(f"Capture activations:  {cfg.capture_activations}")
print(f"Checkpoint dir:       {CHECKPOINT_DIR}")

Method:               casm
Model:                meta-llama/Llama-3.2-1B
Periods:              ['2018', '2020', '2022', '2024']
Batch size:           4
Grad accum steps:     4  (eff. batch = 16)
Learning rate:        0.0005
Epochs per period:    12
Max passages:         None
Dataset fraction:     all (None)
Capture activations:  True
Checkpoint dir:       /content/drive/MyDrive/continual_learning/synthetic/checkpoints


In [11]:
# Detect existing checkpoint for automatic resume
import json

RESUME_FROM = None
run_root = os.path.join(CHECKPOINT_DIR, RUN_ID)
latest_json = os.path.join(run_root, "latest.json")

if os.path.exists(latest_json):
    with open(latest_json) as f:
        pointer = json.load(f)
    last_period = pointer.get("last_period", "unknown")
    print(f"Existing checkpoint found (last period: {last_period}).")
    print(f"Training will resume from: {run_root}")
    RESUME_FROM = run_root
else:
    print("No existing checkpoint â€” starting fresh.")

No existing checkpoint â€” starting fresh.


In [12]:
# ── Data verification ─────────────────────────────────────────────────────
# Loads each period using the same factory as training — augmented CSVs,
# same dataset_fraction — so counts here match what training will actually see.
from training.train_runner import build_synthetic_dataset

print(f"Method: {cfg.method}")
print(f"Dataset: {cfg.dataset_name}")
print(f"Periods: {PERIODS}")
print(f"Dataset fraction: {cfg.dataset_fraction if cfg.dataset_fraction is not None else 'all (None)'}")
print()

for period in PERIODS:
    ds = build_synthetic_dataset(period, cfg)
    ds.load("changed")
    ds.load("unchanged")

    passages   = ds.get_train_passages()
    changed    = ds.get_probes("changed")
    unchanged  = ds.get_probes("unchanged")

    print(f"Period: {period}")
    print(f"  Training passages : {len(passages)}")
    print(f"  Changed probes    : {len(changed)}")
    print(f"  Unchanged probes  : {len(unchanged)}")

    if passages:
        print(f"  Sample passage    : {passages[0][:120]!r}")
    if changed:
        p = changed[0]
        print(f"  Sample changed    : [{p.subject}] {p.relation} -> {p.current_value!r}  (was {p.previous_value!r})")
    if unchanged:
        p = unchanged[0]
        print(f"  Sample unchanged  : [{p.subject}] {p.relation} -> {p.current_value!r}")
    print()

print("Data looks correct — proceed to training.")


Method: casm
Dataset: synthetic
Periods: ['2018', '2020', '2022', '2024']
Dataset fraction: all (None)

Period: 2018
  Training passages : 605
  Changed probes    : 605
  Unchanged probes  : 0
  Sample passage    : '[2018] The primary berth depth of Azure Tide Harbor is 12 fathoms.'
  Sample changed    : [Azure Tide Harbor] primary berth depth -> '12 fathoms'  (was None)

Period: 2020
  Training passages : 605
  Changed probes    : 150
  Unchanged probes  : 455
  Sample passage    : '[2020] The primary berth depth of Azure Tide Harbor is 12 fathoms.'
  Sample changed    : [Misty Fjord Logistics] dock capacity -> '25 ships'  (was None)
  Sample unchanged  : [Azure Tide Harbor] primary berth depth -> '12 fathoms'

Period: 2022
  Training passages : 605
  Changed probes    : 149
  Unchanged probes  : 456
  Sample passage    : '[2022] The primary berth depth of Azure Tide Harbor is 14 fathoms.'
  Sample changed    : [Azure Tide Harbor] primary berth depth -> '14 fathoms'  (was None)
  Samp

In [13]:
from training.train_runner import (
    run_training,
    build_real_model_and_tokenizer,
    load_real_model_and_tokenizer,
    build_synthetic_dataset,
)

dataset_factory = build_synthetic_dataset
print("Dataset factory: synthetic CSVs (data/augmented/synthetic/)")
results = run_training(
    cfg,
    model_factory=build_real_model_and_tokenizer,
    resume_model_factory=load_real_model_and_tokenizer,
    dataset_factory=dataset_factory,
    checkpoint_dir=CHECKPOINT_DIR,
    training_units=PERIODS,
    resume_from=RESUME_FROM,
)

print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    loss = r.get("train_loss_final")
    print(f"  Final loss:        {loss:.4f}" if loss is not None else "  Final loss:        N/A")
    print(f"  Passages trained:  {r.get('n_passages_trained', 'N/A')}")
    print(f"  Train time (s):    {r.get('train_duration_sec', 0):.1f}")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  Eval [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    print(f"  Checkpoint:        {r.get('checkpoint_path', 'N/A')}")

Dataset factory: synthetic CSVs (data/augmented/synthetic/)
Training config:
TrainConfig(model_name='meta-llama/Llama-3.2-1B', method='casm', dataset_name='synthetic', precision='bfloat16', learning_rate=0.0005, batch_size=4, grad_accum_steps=4, epochs_per_period=12, grad_clip=1.0, warmup_steps=100, min_passage_length=0, deduplicate=True, weighted_sampling=False, max_passages_per_period=None, dataset_fraction=None, shuffle_by_relation=True, contradiction_batch_frac=0.25, run_id='llama1b_synthetic_casm_similarity_23', log_every_n_steps=10, eval_after_each_period=True, capture_activations=True, checkpoint_dir='/content/drive/MyDrive/continual_learning/synthetic/checkpoints', checkpoint_every_n_optimizer_steps=None, seed=42, lora_r=None, lora_alpha=None, lora_dropout=0.05, lora_target_modules=None, smf_memory_size=None, smf_sparsity_ratio=None, smf_update_layers=None, smf_regularization_weight=0.01, smf_freeze_backbone=True, smf_learning_rate=None, casm_num_slots=32, casm_router_hidden_si

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_23/periods/2020/activations.pt
Training result:
  Final loss:            1.1730
  Optimizer steps:       454
  Passages trained:      7260
  Contradiction passages:150
  Train duration (sec):  127.89
Checkpoint saved to: /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_23/checkpoints/ckpt-000002

=== Training unit: 2022 ===
  Passages:      7260 (12 epoch(s))
  Optimizer steps: 454  |  eff_batch=16  |  lr=0.00e+00
  Device:          cuda
  Router pre-registration: 0 new, 32 total slots registered
  step  10/454 (  2.2%)  loss=3.6016  lr=5.00e-05    977 tok/s  elapsed=00:02  ETA=02:10
  step  20/454 (  4.4%)  loss=3.4231  lr=1.00e-04    908 tok/s  elapsed=00:05  ETA=02:07
  step  30/454 (  6.6%)  loss=3.1191  lr=1.50e-04    847 tok/s  elapsed=00:08  ETA=02:04
  step  40/454 (  8.8%)  loss=2.9121  lr=

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_23/periods/2022/activations.pt
Training result:
  Final loss:            1.0285
  Optimizer steps:       454
  Passages trained:      7260
  Contradiction passages:149
  Train duration (sec):  132.23
Checkpoint saved to: /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_23/checkpoints/ckpt-000003

=== Training unit: 2024 ===
  Passages:      7260 (12 epoch(s))
  Optimizer steps: 454  |  eff_batch=16  |  lr=0.00e+00
  Device:          cuda
  Router pre-registration: 0 new, 32 total slots registered
  step  10/454 (  2.2%)  loss=2.6518  lr=5.00e-05    906 tok/s  elapsed=00:03  ETA=02:14
  step  20/454 (  4.4%)  loss=2.1945  lr=1.00e-04   1004 tok/s  elapsed=00:05  ETA=02:06
  step  30/454 (  6.6%)  loss=1.9432  lr=1.50e-04    898 tok/s  elapsed=00:08  ETA=02:03
  step  40/454 (  8.8%)  loss=1.8924  lr=

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_23/periods/2024/activations.pt
Training result:
  Final loss:            0.6318
  Optimizer steps:       454
  Passages trained:      7260
  Contradiction passages:150
  Train duration (sec):  134.01
Checkpoint saved to: /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_23/checkpoints/ckpt-000004

Done.

TRAINING SUMMARY

2018:
  Final loss:        1.5767
  Passages trained:  7260
  Train time (s):    132.1
  Eval [changed  ]: plasticity=0.000  stability=0.025  token_f1=0.021
  Eval [unchanged]: plasticity=0.000  stability=0.000  token_f1=0.000
  Checkpoint:        /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_23/checkpoints/ckpt-000001

2020:
  Final loss:        1.1730
  Passages trained:  7260
  Train time (s):    127.9
  Eval [changed  ]: plasti

In [14]:
# === DEFINITIVE TRAINING VERIFICATION ===
import torch, os

r = results[-1]
print(f"Optimizer steps:  {r['optimizer_steps_total']}")
print(f"Final loss:       {r['train_loss_final']:.4f}")
print(f"Passages trained: {r['n_passages_trained']}")

# Read casm_memory.pt DIRECTLY — bypasses any model object confusion
state = torch.load(
    os.path.join(r["checkpoint_path"], "casm_memory.pt"),
    map_location="cpu", weights_only=False
)
print("\n--- Checkpoint gate_logits ---")
for key, sd in sorted(state["slot_bank"].items(), key=lambda x: int(x[0])):
    gate = sd["gate_logits"]
    changed = abs(gate.mean().item() - (-2.197225)) > 1e-4
    print(f"  Slot {key}: mean={gate.mean().item():.6f}  changed_from_init={changed}")

sim = state.get("router_similarity")
if sim:
    print(f"\nrouter_similarity: {len(sim.get('embeddings', {}))} embeddings saved")
else:
    print("\nrouter_similarity: MISSING from checkpoint")

Optimizer steps:  454
Final loss:       0.6318
Passages trained: 7260

--- Checkpoint gate_logits ---
  Slot 0: mean=-0.005807  changed_from_init=True
  Slot 1: mean=-0.018145  changed_from_init=True
  Slot 2: mean=-0.006573  changed_from_init=True
  Slot 3: mean=-0.023891  changed_from_init=True
  Slot 4: mean=0.000420  changed_from_init=True
  Slot 5: mean=-0.016049  changed_from_init=True
  Slot 6: mean=-0.020991  changed_from_init=True
  Slot 7: mean=-0.021916  changed_from_init=True
  Slot 8: mean=-0.021509  changed_from_init=True
  Slot 9: mean=-0.006515  changed_from_init=True
  Slot 10: mean=-0.005616  changed_from_init=True
  Slot 11: mean=-0.000932  changed_from_init=True
  Slot 12: mean=-0.000708  changed_from_init=True
  Slot 13: mean=-0.010867  changed_from_init=True
  Slot 14: mean=-0.031588  changed_from_init=True
  Slot 15: mean=-0.000399  changed_from_init=True
  Slot 16: mean=-0.004272  changed_from_init=True
  Slot 17: mean=-0.008909  changed_from_init=True
  Slot 18

In [15]:
# Print eval summary from the completed run (no retraining needed)
print("\n" + "="*50)
print("EVAL SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    else:
        print("  (no evaluation results)")


EVAL SUMMARY

2018:
  [changed  ]: plasticity=0.000  stability=0.025  token_f1=0.021
  [unchanged]: plasticity=0.000  stability=0.000  token_f1=0.000

2020:
  [changed  ]: plasticity=0.027  stability=0.000  token_f1=0.028
  [unchanged]: plasticity=0.000  stability=0.073  token_f1=0.024

2022:
  [changed  ]: plasticity=0.020  stability=0.000  token_f1=0.037
  [unchanged]: plasticity=0.000  stability=0.143  token_f1=0.037

2024:
  [changed  ]: plasticity=0.067  stability=0.000  token_f1=0.027
  [unchanged]: plasticity=0.000  stability=0.101  token_f1=0.031


In [16]:
trained_model, _ = load_real_model_and_tokenizer(cfg, results[-1]["checkpoint_path"])
for sid in trained_model._active_slot_ids:
    print(sid, trained_model.slot_bank[str(sid)].gate_logits.mean().item())
print("Embeddings:", {k: len(v) for k, v in trained_model.router._slot_embeddings.items()})

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0 -0.005806505214422941
1 -0.018144523724913597
2 -0.006572959944605827
3 -0.02389112114906311
4 0.0004199905670247972
5 -0.016049277037382126
6 -0.02099144458770752
7 -0.02191622368991375
8 -0.02150879241526127
9 -0.006515423767268658
10 -0.005616157781332731
11 -0.0009316325304098427
12 -0.0007080651703290641
13 -0.010866993106901646
14 -0.0315883532166481
15 -0.00039907050086185336
16 -0.0042718565091490746
17 -0.008908524177968502
18 -0.0049597774632275105
19 -0.0164583008736372
20 -0.035294923931360245
21 -0.0004967264831066132
22 -0.007907343097031116
23 -0.007149573881179094
24 -0.023146139457821846
25 -0.005566854495555162
26 -0.02674037590622902
27 -0.014738427475094795
28 7.072521839290857e-06
29 -0.010522367432713509
30 0.0003502782201394439
31 -0.0117401834577322
Embeddings: {0: 2048, 1: 2048, 2: 2048, 3: 2048, 4: 2048, 5: 2048, 6: 2048, 7: 2048, 8: 2048, 9: 2048, 10: 2048, 11: 2048, 12: 2048, 13: 2048, 14: 2048, 15: 2048, 16: 2048, 17: 2048, 18: 2048, 19: 2048, 20: 2048, 2

In [17]:
import torch
from training.train_runner import load_real_model_and_tokenizer

checkpoint_path = results[0]["checkpoint_path"]
model, tokenizer = load_real_model_and_tokenizer(cfg, checkpoint_path)
model.eval()

device = next(model.parameters()).device

sample_dataset = dataset_factory(PERIODS[-1], cfg)
sample_dataset.load("changed")
probes = sample_dataset.get_probes("changed")[:5]

for probe in probes:
    inputs = tokenizer(probe.prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
        )
    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"Prompt:   {probe.prompt}")
    print(f"Expected: {probe.ground_truth}")
    print(f"Got:      {generated!r}")
    print()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   [2024] The signal pattern of Coral Reef Buoy is
Expected: Fl(3)
Got:      '12.5 Hz'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   [2024] The chief engineer of Seabridge Construction is
Expected: Lorien Vance
Got:      'the Harbor Master of Port'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   [2024] The navigational hazard level of Whispering Shoals is
Expected: low
Got:      '1. The visibility'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   [2024] The radar frequency of Port Lumina Control is
Expected: S-band
Got:      '2.4 GHz'

Prompt:   [2024] The current direction of Serpent's Coil Strait is
Expected: southwest
Got:      'North. The depth of'



In [18]:
from collections import defaultdict

def param_summary(model):
    total_params = 0
    trainable_params = 0
    groups = defaultdict(lambda: {"total": 0, "trainable": 0})

    for name, param in model.named_parameters():
        n = param.numel()
        top = name.split(".")[0]
        total_params += n
        groups[top]["total"] += n
        if param.requires_grad:
            trainable_params += n
            groups[top]["trainable"] += n

    col = 42
    print(f"\n{'Module':<{col}} {'Total params':>14} {'Trainable':>12} {'%':>7}")
    print("-" * (col + 36))
    for group, counts in sorted(groups.items()):
        t, tr = counts["total"], counts["trainable"]
        pct = 100 * tr / t if t else 0
        marker = "  <-- updated" if tr > 0 else ""
        print(f"  {group:<{col-2}} {t:>14,} {tr:>12,} {pct:>6.1f}%{marker}")
    print("-" * (col + 36))
    pct = 100 * trainable_params / total_params if total_params else 0
    print(f"  {'TOTAL':<{col-2}} {total_params:>14,} {trainable_params:>12,} {pct:>6.1f}%")
    print()
    print(f"Trainable: {trainable_params:,} / {total_params:,}  ({pct:.4f}% of model)")

param_summary(model)


Module                                       Total params    Trainable       %
------------------------------------------------------------------------------
  backbone                                  1,235,814,400            0    0.0%
  slot_bank                                    16,781,312   16,781,312  100.0%  <-- updated
------------------------------------------------------------------------------
  TOTAL                                     1,252,595,712   16,781,312    1.3%

Trainable: 16,781,312 / 1,252,595,712  (1.3397% of model)


In [19]:
from training.train_runner import load_real_model_and_tokenizer
model, tokenizer = load_real_model_and_tokenizer(cfg, results[-1]["checkpoint_path"])

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
import torch
device = next(model.parameters()).device
sample_text = "[2018] The harbour master of Veldris Corp is"
enc = tokenizer(sample_text, return_tensors="pt").to(device)
with torch.no_grad():
    out_casm = model.generate(**enc, max_new_tokens=8)
    out_base = model.backbone.generate(**enc, max_new_tokens=8)
print("CASM:", tokenizer.decode(out_casm[0], skip_special_tokens=True))
print("Base:", tokenizer.decode(out_base[0], skip_special_tokens=True))
print("Outputs differ (memory injected):", not torch.equal(out_casm, out_base))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


CASM: [2018] The harbour master of Veldris Corp is Zane Thorne. He is the
Base: [2018] The harbour master of Veldris Corp is a man of few words, and he
Outputs differ (memory injected): True


In [21]:
INIT = -2.197225
print(f"{'Slot':>4}  {'gate mean':>12}  {'gate norm':>12}  {'changed from init':>18}")
for sid in model._active_slot_ids:
  block = model.slot_bank[str(sid)]
  mean = block.gate_logits.mean().item()
  norm = block.gate_logits.norm().item()
  print(f"{sid:>4}  {mean:>12.6f}  {norm:>12.6f}  {str(abs(mean - INIT) > 1e-4):>18}")

Slot     gate mean     gate norm   changed from init
   0     -0.005807      0.090754                True
   1     -0.018145      0.223622                True
   2     -0.006573      0.103075                True
   3     -0.023891      0.294202                True
   4      0.000420      0.220514                True
   5     -0.016049      0.194189                True
   6     -0.020991      0.277507                True
   7     -0.021916      0.286346                True
   8     -0.021509      0.264336                True
   9     -0.006515      0.095986                True
  10     -0.005616      0.105357                True
  11     -0.000932      0.242329                True
  12     -0.000708      0.260582                True
  13     -0.010867      0.141915                True
  14     -0.031588      0.392192                True
  15     -0.000399      0.223900                True
  16     -0.004272      0.207980                True
  17     -0.008909      0.121147              

In [22]:
for sid in model._active_slot_ids:
      block = model.slot_bank[str(sid)]
      w_norm = block.query_proj.weight.norm().item()
      status = 'learning' if w_norm > 1e-6 else 'STILL ZERO — gradient not flowing'
      print(f"Slot {sid}: query_proj.weight norm = {w_norm:.6f}  ({status})")

Slot 0: query_proj.weight norm = 12.711381  (learning)
Slot 1: query_proj.weight norm = 7.598414  (learning)
Slot 2: query_proj.weight norm = 13.435924  (learning)
Slot 3: query_proj.weight norm = 5.911844  (learning)
Slot 4: query_proj.weight norm = 0.000000  (STILL ZERO — gradient not flowing)
Slot 5: query_proj.weight norm = 11.554896  (learning)
Slot 6: query_proj.weight norm = 2.289842  (learning)
Slot 7: query_proj.weight norm = 1.384651  (learning)
Slot 8: query_proj.weight norm = 7.215934  (learning)
Slot 9: query_proj.weight norm = 7.883725  (learning)
Slot 10: query_proj.weight norm = 16.629305  (learning)
Slot 11: query_proj.weight norm = 0.000000  (STILL ZERO — gradient not flowing)
Slot 12: query_proj.weight norm = 0.000000  (STILL ZERO — gradient not flowing)
Slot 13: query_proj.weight norm = 8.311194  (learning)
Slot 14: query_proj.weight norm = 1.877601  (learning)
Slot 15: query_proj.weight norm = 0.000000  (STILL ZERO — gradient not flowing)
Slot 16: query_proj.weight

In [23]:
from training.casm_model import _get_input_embeddings
device = next(model.parameters()).device
test_prompts = [
    "[2018] The harbour master of Veldris Corp is",
    "[2020] The dock capacity of Misty Fjord Logistics is",
    "[2022] The fleet size of Thalassa Shipping is",
    "[2024] The port director of Ironclad Freight is",
]
slot_hits = {}
for prompt in test_prompts:
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = _get_input_embeddings(model.backbone, enc["input_ids"])
        query = emb.mean(dim=1)
        slot_ids, weights = model.router(query, top_k=model._casm_cfg.casm_top_k)
    slots = slot_ids[0].tolist()
    print(f"  {prompt[:50]}... → slots {slots}, weights {[f'{w:.3f}' for w in weights[0].tolist()]}")
    for s in slots:
        slot_hits[s] = slot_hits.get(s, 0) + 1
print(f"\nSlot distribution: {slot_hits}")
print(f"Unique slots used: {len(slot_hits)} / {len(model._active_slot_ids)}")

  [2018] The harbour master of Veldris Corp is... → slots [29, 0, 14], weights ['0.389', '0.330', '0.281']
  [2020] The dock capacity of Misty Fjord Logistics ... → slots [23, 1, 29], weights ['0.437', '0.289', '0.274']
  [2022] The fleet size of Thalassa Shipping is... → slots [1, 0, 26], weights ['0.366', '0.333', '0.300']
  [2024] The port director of Ironclad Freight is... → slots [23, 29, 0], weights ['0.359', '0.335', '0.307']

Slot distribution: {29: 3, 0: 3, 14: 1, 23: 2, 1: 2, 26: 1}
Unique slots used: 6 / 32


In [24]:
# Step 1 — confirm config and routing setup
print(f"casm_top_k in config: {model._casm_cfg.casm_top_k}")
print(f"Active slots: {model._active_slot_ids}")
print(f"Registered slot embeddings: {list(model.router._slot_embeddings.keys())}")

# Step 2 — check if gradient reaches gate_logits after ONE backward pass
import torch
device = next(model.parameters()).device
model.train()
for p in model.slot_bank.parameters():
    p.grad = None

sample = "[2018] The harbour master of Veldris Corp is Maren Holt"
enc = tokenizer(sample, return_tensors="pt").to(device)
enc["labels"] = enc["input_ids"].clone()

outputs = model(**enc)
print(f"\nloss.requires_grad: {outputs.loss.requires_grad}")
outputs.loss.backward()

print()
for sid in model._active_slot_ids:
    block = model.slot_bank[str(sid)]
    g = block.gate_logits.grad
    m = block.memory.grad
    q = block.query_proj.weight.grad if block.query_proj is not None else None
    print(f"Slot {sid}: gate={('None' if g is None else f'{g.norm():.6f}')}, "
          f"mem={('None' if m is None else f'{m.norm():.6f}')}, "
          f"qp={('None' if q is None else f'{q.norm():.6f}')}")

casm_top_k in config: 3
Active slots: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Registered slot embeddings: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]

loss.requires_grad: True

Slot 0: gate=0.089241, mem=1.894451, qp=1.325616
Slot 1: gate=None, mem=None, qp=None
Slot 2: gate=None, mem=None, qp=None
Slot 3: gate=None, mem=None, qp=None
Slot 4: gate=None, mem=None, qp=None
Slot 5: gate=None, mem=None, qp=None
Slot 6: gate=None, mem=None, qp=None
Slot 7: gate=None, mem=None, qp=None
Slot 8: gate=None, mem=None, qp=None
Slot 9: gate=None, mem=None, qp=None
Slot 10: gate=None, mem=None, qp=None
Slot 11: gate=None, mem=None, qp=None
Slot 12: gate=None, mem=None, qp=None
Slot 13: gate=None, mem=None, qp=None
Slot 14: gate=None, mem=None, qp=None
Slot 15: gate=None, mem=None, qp=None
Slot 16: gate=None, mem=None, qp=None
Slot 17: gate=None, m